# Exercício 2 — TF-IDF e Ranking com Apache Spark

Neste exercício partimos do índice invertido do Exercício 1 e avançamos para
**ranking de documentos**: calculamos os pesos TF-IDF de cada termo em cada documento
e implementamos um motor de pesquisa simples baseado na **similaridade do coseno**.

O Spark é particularmente adequado para este problema porque o cálculo de TF-IDF
requer múltiplas passagens sobre os dados — em MapReduce seriam jobs separados
com escrita em disco entre eles; em Spark é uma cadeia contínua em memória.

---

### Recapitulação das fórmulas

$$\text{TF}(t, d) = \text{número de vezes que o termo } t \text{ aparece no documento } d$$

$$\text{IDF}(t) = \log\frac{N}{\text{df}(t)} \quad \text{onde } N = \text{total de documentos, } \text{df}(t) = \text{documentos que contêm } t$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

$$\text{sim}(q, d) = \frac{\vec{q} \cdot \vec{d}}{|\vec{d}|} = \frac{\displaystyle\sum_{t \in q} \text{TF-IDF}(t,d)}{\sqrt{\displaystyle\sum_{t} \text{TF-IDF}(t,d)^2}}$$

---
## 1. Setup

In [ ]:
from pyspark import SparkContext, SparkConf
import re
import math

conf = SparkConf().setAppName("IR-TF-IDF").setMaster("local[*]")
sc = SparkContext(conf=conf)
sc.setLogLevel("WARN")

print(f"Spark versão : {sc.version}")
print(f"Modo         : {sc.master}")
print(f"Spark UI     : http://localhost:4040")

---
## 2. Carregar e Parsear a Coleção

In [ ]:
def parse_doc(linha):
    partes = linha.strip().split("\t", 1)
    return (partes[0], partes[1]) if len(partes) == 2 else None

docs = sc.textFile("colecao_grande.txt")
parsed = docs.map(parse_doc).filter(lambda x: x is not None)

# Persistir em memória — vamos usar parsed várias vezes neste notebook
parsed.cache()

N = parsed.count()
print(f"Coleção carregada: {N} documentos")

> **Nota sobre `.cache()`**: como vamos calcular TF, DF e outras métricas a partir de `parsed`,
> o Spark recomputaria o RDD de raiz cada vez. O `.cache()` guarda o resultado em memória
> após a primeira computação, tornando as operações seguintes muito mais rápidas.

---
## 3. Term Frequency (TF)

Para cada par *(documento, termo)*, contamos quantas vezes o termo aparece.

In [ ]:
tf = (
    parsed
    .flatMap(lambda doc: [
        ((doc[0], termo), 1)
        for termo in re.findall(r'\w+', doc[1].lower())
    ])
    .reduceByKey(lambda a, b: a + b)
)
# Formato: ((doc_id, termo), contagem)

print("Exemplos de TF — (doc_id, termo) → contagem:")
for item in tf.take(8):
    print(f"  {str(item[0]):40s} → {item[1]}")

---
## 4. Document Frequency (DF)

Para cada termo, contamos em quantos documentos aparece pelo menos uma vez.

In [ ]:
df = (
    tf
    .map(lambda x: (x[0][1], 1))       # (termo, 1) por cada documento
    .reduceByKey(lambda a, b: a + b)    # (termo, df)
)

print(f"N (total de documentos) = {N}")
print("\nTermos mais comuns na coleção (alto df = baixo poder discriminativo):")
for termo, freq in sorted(df.take(40), key=lambda x: -x[1])[:12]:
    idf = math.log(N / freq)
    print(f"  {termo:25s}  df={freq:3d}  idf={idf:.3f}")

Observa como termos muito frequentes (que aparecem em quase todos os documentos)
têm IDF próximo de zero — são pouco discriminativos para a pesquisa.
Termos raros têm IDF alto e mais peso no ranking.

---
## 5. TF-IDF

Combinamos TF e DF num único `.join()` — a operação que em MapReduce exigiria
dois jobs separados com escrita em disco entre eles.

In [ ]:
# Reorganiza TF para poder fazer join por termo: (termo, (doc_id, tf))
tf_por_termo = tf.map(lambda x: (x[0][1], (x[0][0], x[1])))

tfidf = (
    tf_por_termo
    .join(df)
    # Após o join: (termo, ((doc_id, tf), df))
    .map(lambda x: (
        x[0],                                     # termo
        x[1][0][0],                               # doc_id
        x[1][0][1] * math.log(N / x[1][1])        # tf × log(N/df)
    ))
)
# Formato: (termo, doc_id, score_tfidf)

print("Exemplos de TF-IDF (scores mais altos):")
for termo, doc_id, score in sorted(tfidf.take(60), key=lambda x: -x[2])[:12]:
    print(f"  {doc_id}  {termo:30s}  tfidf = {score:.4f}")

In [ ]:
# Top 5 termos mais discriminativos por documento
top_por_doc = (
    tfidf
    .map(lambda x: (x[1], (x[0], x[2])))            # (doc_id, (termo, score))
    .groupByKey()
    .mapValues(lambda v: sorted(v, key=lambda x: -x[1])[:5])
    .sortByKey()
)

print("Top 5 termos por documento:")
for doc_id, termos in top_por_doc.take(10):
    termos_str = ", ".join([f"{t} ({s:.2f})" for t, s in termos])
    print(f"  {doc_id}: {termos_str}")

---
## 6. Motor de Pesquisa com Similaridade do Coseno

Com os pesos TF-IDF calculados, construímos um índice pesquisável em memória
e implementamos a similaridade do coseno para ranking.

Tratamos cada termo do query com peso 1 (sem TF-IDF do query).
O produto escalar $\vec{q} \cdot \vec{d}$ é a soma dos scores TF-IDF do documento
para os termos presentes no query.
Dividimos pela norma $|\vec{d}|$ para não penalizar documentos curtos.

In [ ]:
# Índice TF-IDF em memória: {termo: [(doc_id, score), ...]}
indice_tfidf = (
    tfidf
    .map(lambda x: (x[0], (x[1], x[2])))
    .groupByKey()
    .mapValues(list)
    .collectAsMap()
)

# Norma de cada documento: sqrt( Σ score² )
normas = (
    tfidf
    .map(lambda x: (x[1], x[2] ** 2))
    .reduceByKey(lambda a, b: a + b)
    .mapValues(math.sqrt)
    .collectAsMap()
)

print(f"Índice TF-IDF : {len(indice_tfidf)} termos distintos")
print(f"Normas        : {len(normas)} documentos")

In [ ]:
def pesquisar(query, top_k=5):
    """Devolve os top_k documentos mais relevantes para o query dado,
    usando similaridade do coseno com pesos TF-IDF."""
    termos_query = re.findall(r'\w+', query.lower())

    # Produto escalar: acumula TF-IDF para cada doc nos termos do query
    scores = {}
    for termo in termos_query:
        if termo in indice_tfidf:
            for doc_id, score in indice_tfidf[termo]:
                scores[doc_id] = scores.get(doc_id, 0) + score

    # Normalizar pela norma do documento
    coseno = {
        doc_id: score / normas.get(doc_id, 1)
        for doc_id, score in scores.items()
    }

    return sorted(coseno.items(), key=lambda x: -x[1])[:top_k]

In [ ]:
queries = [
    "índice invertido",
    "MapReduce Spark distribuído",
    "ranking relevância TF-IDF",
    "PageRank links web",
    "clustering documentos",
]

for q in queries:
    resultados = pesquisar(q)
    print(f"Query: '{q}'")
    for doc_id, score in resultados:
        print(f"  {doc_id}  coseno = {score:.4f}")
    print()

---
## 7. Experimenta

Testa os teus próprios queries. Experimenta queries com um só termo, com vários termos,
com termos que não existem na coleção, e observa como o ranking muda.

In [ ]:
meu_query = "recuperação informação modelo vetorial coseno"

resultados = pesquisar(meu_query, top_k=10)
print(f"Query: '{meu_query}'\n")
for doc_id, score in resultados:
    print(f"  {doc_id}  coseno = {score:.4f}")

---
## 8. Terminar

In [ ]:
sc.stop()
print("SparkContext terminado.")